# Notebook 03 — The Curl (∇×F)

Curl turns a **vector field** into a **vector field**. The curl vector points
along the local axis of rotation (right-hand rule), and its magnitude
measures *twice* the local angular velocity of the fluid.

In 3D:

$$\nabla \times \mathbf{F} = \left(\frac{\partial F_z}{\partial y} - \frac{\partial F_y}{\partial z},\ \frac{\partial F_x}{\partial z} - \frac{\partial F_z}{\partial x},\ \frac{\partial F_y}{\partial x} - \frac{\partial F_x}{\partial y}\right)$$

In **2D** (when F lives in the xy-plane), only the z-component is non-zero,
and we usually treat the curl as a **scalar field**:

$$(\nabla \times \mathbf{F})_z = \frac{\partial F_y}{\partial x} - \frac{\partial F_x}{\partial y}$$

Positive = counter-clockwise rotation. Negative = clockwise.


In [ ]:
# Make src/ importable from inside notebooks/
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
%matplotlib inline

from src import operators as ops
from src import visualizers as viz


## 1. By hand — pure rotation

Let $\mathbf{F}(x, y) = (-y, x)$. This is the **rigid rotation field**: every
point moves with a velocity tangent to a circle around the origin.

- $\dfrac{\partial F_y}{\partial x} = \dfrac{\partial x}{\partial x} = 1$
- $\dfrac{\partial F_x}{\partial y} = \dfrac{\partial(-y)}{\partial y} = -1$
- Curl: $1 - (-1) = 2$  *(constant, positive — CCW everywhere)*

The full 3D curl is $\nabla \times (-y, x, 0) = (0, 0, 2)$ — a vector pointing
along $+z$, consistent with the right-hand rule for CCW rotation in the
xy-plane.

## 2. By hand — a shear field

Let $\mathbf{F}(x, y) = (y, 0)$. This is uniform horizontal flow whose speed
grows with height — a *shear*.

- $\dfrac{\partial F_y}{\partial x} = 0$
- $\dfrac{\partial F_x}{\partial y} = 1$
- Curl: $0 - 1 = -1$

Negative everywhere — clockwise rotation. **Counter-intuitive, because the
streamlines are horizontal lines, not circles.** But imagine a tiny paddle
wheel placed in the flow: the top of the wheel is in faster current than the
bottom, so the wheel rotates clockwise. That's what curl measures — *local*
rotation of the fluid, not global path shape.


## 3. SymPy verification (full 3D)

In [ ]:
x, y, z = sp.symbols('x y z')
Fx_sym, Fy_sym, Fz_sym = -y, x, 0

curl_x = sp.diff(Fz_sym, y) - sp.diff(Fy_sym, z)
curl_y = sp.diff(Fx_sym, z) - sp.diff(Fz_sym, x)
curl_z = sp.diff(Fy_sym, x) - sp.diff(Fx_sym, y)

print(f"∇×F = ({curl_x}, {curl_y}, {curl_z})")


## 4. NumPy on a grid

In [ ]:
x_vals = np.linspace(-2, 2, 200)
y_vals = np.linspace(-2, 2, 200)
X, Y = np.meshgrid(x_vals, y_vals)
dx, dy = x_vals[1] - x_vals[0], y_vals[1] - y_vals[0]

# Rotation field
Fx_rot, Fy_rot = -Y, X
curl_rot = ops.curl_2d(Fx_rot, Fy_rot, dx, dy)
print(f"Mean curl of rotation field: {np.mean(curl_rot):.4f}  (expect 2.0)")

# Shear field
Fx_sh, Fy_sh = Y, np.zeros_like(Y)
curl_sh = ops.curl_2d(Fx_sh, Fy_sh, dx, dy)
print(f"Mean curl of shear field:    {np.mean(curl_sh):.4f}  (expect -1.0)")


## 5. Visualization

In [ ]:
fig = viz.plot_vector_with_curl(
    X, Y, Fx_rot, Fy_rot, curl_rot,
    title=r"Rotation field $\mathbf{F} = (-y, x)$  —  curl $= 2$",
)
plt.show()

fig = viz.plot_vector_with_curl(
    X, Y, Fx_sh, Fy_sh, curl_sh,
    title=r"Shear field $\mathbf{F} = (y, 0)$  —  curl $= -1$",
)
plt.show()


**Reading these plots.**

- *Rotation field*: streamlines are concentric circles; the curl heatmap is
  uniformly green (positive, CCW) across the whole domain.
- *Shear field*: streamlines are straight horizontal lines, but the heatmap
  is uniformly magenta (negative, CW). This is the paddle-wheel argument
  visualized — straight lines can still carry rotation.


## 6. An irrotational example

Let $\mathbf{F} = (x, y)$ — the radial source from the divergence notebook.

- $\dfrac{\partial F_y}{\partial x} = 0$, $\dfrac{\partial F_x}{\partial y} = 0$
- Curl $= 0$

The field has plenty of divergence (it's full of sources) but zero curl —
no swirl anywhere. **Divergence and curl are independent.** A field can have
either, both, or neither.


In [ ]:
Fx_rad, Fy_rad = X, Y
curl_rad = ops.curl_2d(Fx_rad, Fy_rad, dx, dy)
print(f"Max |curl| of radial field: {np.max(np.abs(curl_rad)):.2e}")


## 7. Exercise

Compute the curl by hand for $\mathbf{F}(x, y) = (-y/r^2,\ x/r^2)$ where
$r = \sqrt{x^2 + y^2}$. This is the *vortex* field from fluid dynamics — what
do you expect, and what do you get?

(Spoiler: you get zero everywhere except at the origin, where the field is
singular. This is a famous example showing that "curl-free" doesn't always
mean "no rotation" — line integrals around the origin still pick up
$2\pi$. Look up *Stokes' theorem* and *simply-connected domains* if you
want to go deeper.)
